#  Hyperparameter Tuning  and Sklearn baseline

In [ ]:
#  Hyperparameter Tuning

from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder
from pyspark.ml.classification import (
    LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
)

print("Starting tuning with TrainValidationSplit.")

initial_results = {
    "LogisticRegression": lr_acc,
    "DecisionTree":       dt_acc,
    "RandomForest":       rf_acc,
}
best_model_name = max(initial_results, key=initial_results.get)
print("Best model selected for tuning:", best_model_name)
print("Initial accuracy:", round(initial_results[best_model_name], 4))

# 5% subsample - enough to tune reliably on CPU without hanging
train_cv = train_df.sample(False, 0.05, seed=42).persist()
train_cv.count()
print("Tuning subset row count:", train_cv.count())

# Checkpoint forces materialisation to disk so the 2 fits
train_cv.checkpoint()
train_cv.count()

if best_model_name == "RandomForest":
    tuned_base = RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        maxDepth=10,
        maxBins=32,
        minInstancesPerNode=10,
        featureSubsetStrategy="sqrt",
        seed=42
    )
    param_grid = (ParamGridBuilder()
        .addGrid(tuned_base.numTrees, [50, 100])
        .build()
    )

elif best_model_name == "DecisionTree":
    tuned_base = DecisionTreeClassifier(
        featuresCol="features",
        labelCol="label",
        maxBins=32,
        seed=42
    )
    param_grid = (ParamGridBuilder()
        .addGrid(tuned_base.maxDepth, [10, 13])
        .build()
    )

else:
    tuned_base = LogisticRegression(
        featuresCol="features",
        labelCol="label",
        family="multinomial",
        maxIter=30
    )
    param_grid = (ParamGridBuilder()
        .addGrid(tuned_base.regParam, [0.001, 0.01])
        .build()
    )

# TrainValidationSplit: 80% train, 20% validation, 2 fits total, no folds
tvs = TrainValidationSplit(
    estimator=tuned_base,
    estimatorParamMaps=param_grid,
    evaluator=acc_eval,
    trainRatio=0.8,
    parallelism=1
)

start_cv = time.perf_counter()
tvs_model = tvs.fit(train_cv)
cv_time   = time.perf_counter() - start_cv

train_cv.unpersist()
print("Tuning subset unpersisted.")

best_tuned_model = tvs_model.bestModel
tuned_pred       = best_tuned_model.transform(test_df)
tuned_acc        = float(acc_eval.evaluate(tuned_pred))
tuned_f1         = float(f1_eval.evaluate(tuned_pred))

print("Tuned model:", best_model_name)
print("Tuned accuracy:", round(tuned_acc, 4))
print("Tuned F1 score:", round(tuned_f1, 4))
print("Tuning time (seconds):", round(cv_time, 2))
print("Improvement over baseline:", round(tuned_acc - initial_results[best_model_name], 4))

best_tuned_model.write().overwrite().save(MODEL_DIR + "/best_tuned_model")
tuned_pred.write.mode("overwrite").parquet(PRED_DIR + "/tuned_predictions")
print("Tuned model and predictions saved.")

# Update rf_pred so all downstream sections use the tuned version
if best_model_name == "RandomForest":
    rf_pred  = tuned_pred
    rf_acc   = tuned_acc
    rf_f1    = tuned_f1
    rf_model = best_tuned_model
elif best_model_name == "DecisionTree":
    dt_pred = tuned_pred
    dt_acc  = tuned_acc
    dt_f1   = tuned_f1
elif best_model_name == "LogisticRegression":
    lr_pred = tuned_pred
    lr_acc  = tuned_acc
    lr_f1   = tuned_f1

print("Best pred variable updated for downstream sections.")

Starting tuning with TrainValidationSplit.
Best model selected for tuning: RandomForest
Initial accuracy: 0.6108
Tuning subset row count: 48041
Tuning subset unpersisted.
Tuned model: RandomForest
Tuned accuracy: 0.6029
Tuned F1 score: 0.4869
Tuning time (seconds): 126.46
Improvement over baseline: -0.0079
Tuned model and predictions saved.
Best pred variable updated for downstream sections.


In [ ]:
PLOT_DIR = "/content/drive/MyDrive/chicago_plots"
os.makedirs(PLOT_DIR, exist_ok=True)
print("Plot directory ready:", PLOT_DIR)

Plot directory ready: /content/drive/MyDrive/chicago_plots


In [ ]:
# Confusion Matrix Plots for LR, DT, RF
# Row-normalised to percentages so imbalanced classes are still readable

lr_pred = lr_pred.persist(); lr_pred.count()
dt_pred = dt_pred.persist(); dt_pred.count()
rf_pred = rf_pred.persist(); rf_pred.count()


# Cast to int - PySpark label and prediction columns are Double by default
lr_pd = lr_pred.select(
    F.col("label").cast("int").alias("label"),
    F.col("prediction").cast("int").alias("prediction")
).toPandas()

dt_pd = dt_pred.select(
    F.col("label").cast("int").alias("label"),
    F.col("prediction").cast("int").alias("prediction")
).toPandas()

rf_pd = rf_pred.select(
    F.col("label").cast("int").alias("label"),
    F.col("prediction").cast("int").alias("prediction")
).toPandas()

label_names = label_model.labels
n_classes   = len(label_names)
print("Label names:", label_names)
print("LR test rows:", len(lr_pd))
print("DT test rows:", len(dt_pd))
print("RF test rows:", len(rf_pd))

def plot_confusion_matrix(y_true, y_pred, labels, model_name, save_path):
    y_true = np.array(y_true).astype(int)
    y_pred = np.array(y_pred).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))

    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    cm_pct = cm.astype(float) / row_sums * 100

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(
        cm_pct,
        annot=True,
        fmt=".1f",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
        linewidths=0.5,
        ax=ax,
        vmin=0,
        vmax=100
    )
    ax.set_xlabel("Predicted Label", fontsize=11)
    ax.set_ylabel("Actual Label", fontsize=11)
    ax.set_title(model_name + " Confusion Matrix (row %)", fontsize=12)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", save_path)

plot_confusion_matrix(
    lr_pd["label"], lr_pd["prediction"], label_names,
    "Logistic Regression",
    PLOT_DIR + "/cm_logistic_regression.png"
)

plot_confusion_matrix(
    dt_pd["label"], dt_pd["prediction"], label_names,
    "Decision Tree",
    PLOT_DIR + "/cm_decision_tree.png"
)

plot_confusion_matrix(
    rf_pd["label"], rf_pd["prediction"], label_names,
    "Random Forest (Tuned)",
    PLOT_DIR + "/cm_random_forest.png"
)


Label names: ['PROPERTY', 'VIOLENT', 'DRUG']
LR test rows: 241315
DT test rows: 241315
RF test rows: 241315
Saved: /content/drive/MyDrive/chicago_plots/cm_logistic_regression.png
Saved: /content/drive/MyDrive/chicago_plots/cm_decision_tree.png
Saved: /content/drive/MyDrive/chicago_plots/cm_random_forest.png


In [ ]:
# ROC Curves
# Multi-class: one-vs-rest per class for LR, DT, RF
# Binary: single curve for GBT


def collect_probs(pred_df):
    pdf = pred_df.select(
        F.col("label").cast("int").alias("label"),
        vector_to_array(F.col("probability")).alias("prob_arr")
    ).toPandas()
    y_true      = pdf["label"].values.astype(int)
    prob_matrix = np.array(pdf["prob_arr"].tolist(), dtype=float)
    return y_true, prob_matrix

lr_y, lr_probs = collect_probs(lr_pred)
dt_y, dt_probs = collect_probs(dt_pred)
rf_y, rf_probs = collect_probs(rf_pred)

print("Probability shapes - LR:", lr_probs.shape,
      "| DT:", dt_probs.shape, "| RF:", rf_probs.shape)

classes_list = list(range(n_classes))
y_bin_lr     = label_binarize(lr_y.astype(int), classes=classes_list)
y_bin_dt     = label_binarize(dt_y.astype(int), classes=classes_list)
y_bin_rf     = label_binarize(rf_y.astype(int), classes=classes_list)

class_colors = ["steelblue", "darkorange", "seagreen"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

model_configs = [
    ("Logistic Regression", y_bin_lr, lr_probs),
    ("Decision Tree",       y_bin_dt, dt_probs),
    ("Random Forest",       y_bin_rf, rf_probs),
]

for ax, (model_name, y_bin, probs) in zip(axes, model_configs):
    for i in range(n_classes):
        if y_bin[:, i].sum() == 0:
            print("Skipping class", i, "no positive samples")
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
        roc_auc     = round(auc(fpr, tpr), 3)
        ax.plot(
            fpr, tpr,
            color=class_colors[i],
            lw=2,
            label=str(label_names[i]) + "  AUC=" + str(roc_auc)
        )
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random baseline")
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.02])
    ax.set_xlabel("False Positive Rate", fontsize=10)
    ax.set_ylabel("True Positive Rate", fontsize=10)
    ax.set_title(model_name + " ROC", fontsize=11)
    ax.legend(loc="lower right", fontsize=8)

plt.suptitle("ROC Curves - One vs Rest per Class", fontsize=13)
plt.tight_layout()
fig.savefig(PLOT_DIR + "/roc_curves_multiclass.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved: roc_curves_multiclass.png")

# GBT Binary ROC
print("Loading GBT predictions for binary ROC...")
gbt_saved = spark.read.parquet(PRED_DIR + "/gbt_predictions")
print("GBT columns:", gbt_saved.columns)

gbt_pd = gbt_saved.select(
    F.col("label_binary").cast("double").alias("label"),
    vector_to_array(F.col("probability")).alias("prob_arr")
).toPandas()

gbt_y_true = gbt_pd["label"].values.astype(int)
gbt_prob_1 = np.array(gbt_pd["prob_arr"].tolist(), dtype=float)[:, 1]

fpr_gbt, tpr_gbt, _ = roc_curve(gbt_y_true, gbt_prob_1)
auc_gbt = round(auc(fpr_gbt, tpr_gbt), 3)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr_gbt, tpr_gbt, color="crimson", lw=2,
        label="VIOLENT vs Other  AUC=" + str(auc_gbt))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random baseline")
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.02])
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title("GBT Binary Classifier ROC", fontsize=12)
ax.legend(loc="lower right", fontsize=10)
plt.tight_layout()
fig.savefig(PLOT_DIR + "/roc_curve_gbt_binary.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved: roc_curve_gbt_binary.png")
print("GBT AUC:", auc_gbt)

Probability shapes - LR: (241315, 4) | DT: (241315, 4) | RF: (241315, 4)
Saved: roc_curves_multiclass.png
Loading GBT predictions for binary ROC...
GBT columns: ['ID', 'Year', 'year_num', 'Primary Type', 'crime_category', 'crime_group', 'Arrest', 'Latitude', 'Longitude', 'hour', 'day_of_week', 'month', 'District', 'Beat', 'is_night', 'is_evening', 'is_business_hours', 'is_weekend', 'season', 'label', 'features', 'label_binary', 'rawPrediction', 'probability', 'prediction']
Saved: roc_curve_gbt_binary.png
GBT AUC: 0.628


In [ ]:
# Scikit-learn Baseline vs PySpark MLlib

import sklearn
from sklearn.linear_model import LogisticRegression as SklearnLR
from sklearn.tree import DecisionTreeClassifier as SklearnDT
from sklearn.ensemble import RandomForestClassifier as SklearnRF
from sklearn.metrics import accuracy_score, f1_score as sk_f1_score
from pyspark.ml.functions import vector_to_array

print("Extracting 5 percent sample to pandas for sklearn...")

def spark_to_numpy(spark_df, frac, seed):
    pdf = spark_df.sample(False, frac, seed=seed).select(
        vector_to_array("features").alias("feat"),
        F.col("label").cast("int").alias("label")
    ).toPandas()
    X = np.array(pdf["feat"].tolist(), dtype=float)
    y = pdf["label"].values.astype(int)
    return X, y

X_train_sk, y_train_sk = spark_to_numpy(train_df, 0.05, 42)
X_test_sk,  y_test_sk  = spark_to_numpy(test_df,  0.10, 42)

print("Sklearn train shape:", X_train_sk.shape)
print("Sklearn test shape: ", X_test_sk.shape)

sklearn_results = []

# Logistic Regression
start = time.perf_counter()
sk_lr = SklearnLR(max_iter=200, random_state=42, multi_class="multinomial", n_jobs=-1)
sk_lr.fit(X_train_sk, y_train_sk)
sk_lr_time = time.perf_counter() - start
sk_lr_pred = sk_lr.predict(X_test_sk)
sk_lr_acc  = round(accuracy_score(y_test_sk, sk_lr_pred), 4)
sk_lr_f1   = round(sk_f1_score(y_test_sk, sk_lr_pred, average="weighted"), 4)
print("Sklearn LR   | Acc:", sk_lr_acc, "| F1:", sk_lr_f1, "| Time:", round(sk_lr_time, 2), "s")
sklearn_results.append({
    "model": "Sklearn LogisticRegression", "framework": "Sklearn (single-node)",
    "accuracy": sk_lr_acc, "f1_score": sk_lr_f1,
    "train_time_sec": round(sk_lr_time, 2)
})

# Decision Tree
start = time.perf_counter()
sk_dt = SklearnDT(max_depth=15, random_state=42)
sk_dt.fit(X_train_sk, y_train_sk)
sk_dt_time = time.perf_counter() - start
sk_dt_pred = sk_dt.predict(X_test_sk)
sk_dt_acc  = round(accuracy_score(y_test_sk, sk_dt_pred), 4)
sk_dt_f1   = round(sk_f1_score(y_test_sk, sk_dt_pred, average="weighted"), 4)
print("Sklearn DT   | Acc:", sk_dt_acc, "| F1:", sk_dt_f1, "| Time:", round(sk_dt_time, 2), "s")
sklearn_results.append({
    "model": "Sklearn DecisionTree", "framework": "Sklearn (single-node)",
    "accuracy": sk_dt_acc, "f1_score": sk_dt_f1,
    "train_time_sec": round(sk_dt_time, 2)
})

# Random Forest
start = time.perf_counter()
sk_rf = SklearnRF(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
sk_rf.fit(X_train_sk, y_train_sk)
sk_rf_time = time.perf_counter() - start
sk_rf_pred = sk_rf.predict(X_test_sk)
sk_rf_acc  = round(accuracy_score(y_test_sk, sk_rf_pred), 4)
sk_rf_f1   = round(sk_f1_score(y_test_sk, sk_rf_pred, average="weighted"), 4)
print("Sklearn RF   | Acc:", sk_rf_acc, "| F1:", sk_rf_f1, "| Time:", round(sk_rf_time, 2), "s")
sklearn_results.append({
    "model": "Sklearn RandomForest", "framework": "Sklearn (single-node)",
    "accuracy": sk_rf_acc, "f1_score": sk_rf_f1,
    "train_time_sec": round(sk_rf_time, 2)
})

sklearn_df = pd.DataFrame(sklearn_results)
sklearn_df.to_csv(OUT_DIR + "/dash2_sklearn_baseline.csv", index=False)
print("Saved: dash2_sklearn_baseline.csv")

# Side by side comparison table
comparison_full = pd.DataFrame([
    {"framework": "PySpark MLlib", "model": "Logistic Regression",
     "accuracy": round(lr_acc, 4), "f1_score": round(lr_f1, 4),
     "train_time_sec": round(lr_time, 2), "data_size": "full dataset"},
    {"framework": "PySpark MLlib", "model": "Decision Tree",
     "accuracy": round(dt_acc, 4), "f1_score": round(dt_f1, 4),
     "train_time_sec": round(dt_time, 2), "data_size": "full dataset"},
    {"framework": "PySpark MLlib", "model": "Random Forest",
     "accuracy": round(rf_acc, 4), "f1_score": round(rf_f1, 4),
     "train_time_sec": round(rf_time, 2), "data_size": "full dataset"},
    {"framework": "PySpark MLlib", "model": "GBT (Binary)",
     "accuracy": round(gbt_acc, 4), "f1_score": round(gbt_f1, 4),
     "train_time_sec": round(gbt_time, 2), "data_size": "full dataset"},
    {"framework": "PySpark MLlib", "model": best_model_name + " Tuned",
     "accuracy": round(tuned_acc, 4), "f1_score": round(tuned_f1, 4),
     "train_time_sec": round(cv_time, 2), "data_size": "5pct subsample"},
    {"framework": "Sklearn (single-node)", "model": "Logistic Regression",
     "accuracy": sk_lr_acc, "f1_score": sk_lr_f1,
     "train_time_sec": round(sk_lr_time, 2), "data_size": "5pct subsample"},
    {"framework": "Sklearn (single-node)", "model": "Decision Tree",
     "accuracy": sk_dt_acc, "f1_score": sk_dt_f1,
     "train_time_sec": round(sk_dt_time, 2), "data_size": "5pct subsample"},
    {"framework": "Sklearn (single-node)", "model": "Random Forest",
     "accuracy": sk_rf_acc, "f1_score": sk_rf_f1,
     "train_time_sec": round(sk_rf_time, 2), "data_size": "5pct subsample"},
])
comparison_full.to_csv(OUT_DIR + "/dash2_mllib_vs_sklearn.csv", index=False)
print("Saved: dash2_mllib_vs_sklearn.csv")
print(comparison_full.to_string(index=False))

Extracting 5 percent sample to pandas for sklearn...
Sklearn train shape: (48041, 22)
Sklearn test shape:  (23934, 22)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Sklearn LR   | Acc: 0.5994 | F1: 0.4563 | Time: 6.93 s
Sklearn DT   | Acc: 0.5613 | F1: 0.5301 | Time: 0.98 s
Sklearn RF   | Acc: 0.602 | F1: 0.53 | Time: 12.42 s
Saved: dash2_sklearn_baseline.csv
Saved: dash2_mllib_vs_sklearn.csv
            framework               model  accuracy  f1_score  train_time_sec      data_size
        PySpark MLlib Logistic Regression    0.5992    0.4527           75.58   full dataset
        PySpark MLlib       Decision Tree    0.5984    0.5316           52.20   full dataset
        PySpark MLlib       Random Forest    0.6029    0.4869         4004.04   full dataset
        PySpark MLlib        GBT (Binary)    0.6931    0.5902          258.78   full dataset
        PySpark MLlib  RandomForest Tuned    0.6029    0.4869          126.46 5pct subsample
Sklearn (single-node) Logistic Regression    0.5994    0.4563            6.93 5pct subsample
Sklearn (single-node)       Decision Tree    0.5613    0.5301            0.98 5pct subsample
Sklearn (single-node)    